## Accessing Vectors with LatinCy

In [1]:
# Imports & setup

import spacy
import numpy as np
from tabulate import tabulate
from pprint import pprint
nlp = spacy.load('la_core_web_lg')
text = "Haec narrantur a poetis de Perseo. Perseus filius erat Iovis, maximi deorum; avus eius Acrisius appellabatur."
doc = nlp(text)
print(doc)

Haec narrantur a poetis de Perseo . Perseus filius erat Iouis , maximi deorum ; auus eius Acrisius appellabatur .


<!-- Word tokenization is the task of splitting a text into words (and wordlike units like punctuation, numbers, etc.). For the LatinCy models, tokenization is the fundamental pipeline component on which all other components depend. SpaCy uses non-destructive, "canonical" tokenization, i.e. non-destructive, in that the original text can be untokenized, so to speak, based on `Token` annotations and canonical in that indices are assigned to each token during this process and these indices are used to refer to the tokens in other annotations. (Tokens *can* be separated or merged, but this requires the user to actively undo and redefine the tokenization output.) LatinCy uses a modified version of the default spaCy tokenizer that recognizes and splits enlitic *-que* using a rules-based process. (NB: It is in the LatinCy [development plan](projects.qmd#enclitics-project) to move enclitic splitting to a separate post-tokenization component.)

The spaCy `Doc` object is an iterable and tokens are the iteration unit. -->

Vectors are used in two different ways in LatinCy. First, a static embedding layer is used during training to initialize what will become a contextual embedding layer during training. Second, this contextual embedding layer—spaCy's [tok2vec](https://spacy.io/api/tok2vec) component—is used to produce the final token vectors that are used as input features for other components. LatinCy models use Floret vectors for the static embedding layer, both in the [md](https://huggingface.co/latincy/la_vectors_floret_md) and [lg](https://huggingface.co/latincy/la_vectors_floret_lg) model sizes. The [sm](https://huggingface.co/latincy/la_vectors_floret_sm) model size does not use static vectors at all, relying instead on hash embeddings of word-level features (normalized form, prefix, suffix, shape) to produce the contextual tok2vec output.

Again...

1. **Static floret vectors** (300 dimensions) — trained on the LatinCy texts corpus, these assign a fixed vector to each word form. Accessed via `token.vector`. The same word always receives the same vector, regardless of context.

2. **Contextual tok2vec vectors** (96 dimensions) — produced by a CNN that takes the static vectors as input and incorporates surrounding context. Accessed via `doc.tensor[token.i]`. The same word receives *different* vectors in different sentences.

**New for v3.9**: The [`pretrain`](https://spacy.io/usage/embeddings-transformers#pretraining) initializes the model with information from a large amount of raw text so that the model can learn sentence structure, word cooccurrence, and other general language patterns.

Here is the vector for the first token 'Haec' in the text given above, output trunctated to the first 10 dimensions:

In [2]:
tokens = [token for token in doc]
for token in tokens:
    print(token.text)
    print(token.vector[:10], "etc.")
    break

Haec
[-0.7778918  -0.24657087 -2.321399   -4.946182   -0.47471917 -1.3058611
 -0.06238138 -1.4708582  -2.1147673  -2.00264   ] etc.


And here is a list of the first 10 tokens, each with the first 10 dimensions of their static floret vector...

In [3]:
vector_example_data = []
for token in tokens[:10]:
    vector_example_data.append((token.text, *token.vector[:10]))

print(tabulate(vector_example_data))

---------  ---------  ---------  ---------  ---------  ----------  ---------  ----------  -----------  ---------  ----------
Haec       -0.777892  -0.246571  -2.3214    -4.94618   -0.474719   -1.30586   -0.0623814  -1.47086     -2.11477   -2.00264
narrantur   1.44294    2.76409   -0.174809  -1.1091     1.32203     1.68776    1.97994     1.67613     -0.558864  -1.3482
a           3.0887     4.14845    4.40425   12.6695     3.4945      0.35588    0.8834      2.27109      4.9266    -0.78953
poetis      2.54862    2.26177    2.09329   -0.416738   0.0676873   2.26123    1.91584     1.16091      0.548886   0.563511
de         -2.47883    8.23127    2.65751   -0.477113  -0.854738   -3.52511   -0.143562    4.56366      5.87061   -1.26645
Perseo      0.134013   0.504851  -2.4602     1.95082   -2.07618     0.364746  -2.08651     0.00902766  -2.2649     0.958519
.           1.91543   10.3048    -8.6075    -3.56385    9.61896     3.71153   -4.3554     -4.69565      5.341      2.3426
Perseus    -0.

We can get the vector length from the `shape` attribute...

In [4]:
print(doc[0].vector.shape)

(300,)


### Static vectors are context-independent

Because `token.vector` returns static floret vectors, the same word form always produces the same vector — regardless of the sentence it appears in. Here we compare the vector for *erat* in two different sentences:

In [5]:
# Static floret vectors: same word → same vector
doc_a = nlp("Caesar magnus erat.")
doc_b = nlp("Tempus longum erat.")

erat_a = [t for t in doc_a if t.text == "erat"][0]
erat_b = [t for t in doc_b if t.text == "erat"][0]

print(f"'erat' in '{doc_a.text}':")
print(erat_a.vector[:10])
print(f"\n'erat' in '{doc_b.text}':")
print(erat_b.vector[:10])
print(f"\nVectors identical? {np.array_equal(erat_a.vector, erat_b.vector)}")

'erat' in 'Caesar magnus erat .':
[ 5.0704026   1.0435151  -0.56230766  3.801923   -1.5406077  -2.3206182
 -2.4345021   1.8656712   4.334582   -0.74421906]

'erat' in 'Tempus longum erat .':
[ 5.0704026   1.0435151  -0.56230766  3.801923   -1.5406077  -2.3206182
 -2.4345021   1.8656712   4.334582   -0.74421906]

Vectors identical? True


### Contextual vectors from tok2vec

The tok2vec component on the other hand takes the static floret vectors as input and produces *contextual* representations that incorporate information from surrounding tokens. These are stored in `doc.tensor`, a matrix with one row per token, and have 96 dimensions in the `lg` model.

Unlike the static vectors, the contextual tok2vec output for the same word form *differs* depending on context, though note still the high similarity between the two vectors for *erat* in the different sentences (cosine similarity of ~0.97).    

In [6]:
# Contextual tok2vec output: same word → different vector in different context
tensor_a = doc_a.tensor[erat_a.i]
tensor_b = doc_b.tensor[erat_b.i]

print(f"tok2vec output for 'erat' in '{doc_a.text}':")
print(tensor_a[:10])
print(f"\ntok2vec output for 'erat' in '{doc_b.text}':")
print(tensor_b[:10])
print(f"\nVectors identical? {np.array_equal(tensor_a, tensor_b)}")
cos_sim = np.dot(tensor_a, tensor_b) / (np.linalg.norm(tensor_a) * np.linalg.norm(tensor_b))
print(f"Cosine similarity: {cos_sim:.4f}")

tok2vec output for 'erat' in 'Caesar magnus erat .':
[-0.7828857   1.2530129   3.217065   -1.987933    1.0486498  -0.9898325
 -2.239715    0.518736   -0.5815717  -0.19825797]

tok2vec output for 'erat' in 'Tempus longum erat .':
[ 0.6583333   1.0896324   3.5292163   0.29009092  1.1463788  -1.0406121
 -1.8273878   0.71445364  1.443218   -1.7423413 ]

Vectors identical? False
Cosine similarity: 0.7458


### Span and document vectors

The `span.vector` and `doc.vector` properties return the mean of the *static* `token.vector` values — not the contextual tok2vec output. These averaged static vectors provide a simple, deterministic representation for a span of text:

In [7]:
sents = [sent for sent in doc.sents]

for sent in sents:
    print(sent)
    print(sent.vector[:10])
    break

Haec narrantur a poetis de Perseo .
[ 0.83899707  3.9955199  -0.62983674  0.58676285  1.5853631   0.5071678
 -0.2669535   0.50204414  1.6783661  -0.22031358]


The `doc.similarity()` method computes cosine similarity between averaged static vectors. Because these are based on static floret embeddings, the similarity score reflects overlap in word forms and their trained associations, not context-dependent meaning:

In [8]:
text_1 = "Omnia vincit amor."
text_2 = "Omnia vincit amicitia."

doc_1 = nlp(text_1)
doc_2 = nlp(text_2)

print("\nSimilarity between the two sentences...")
similarity = doc_1.similarity(doc_2)
print(similarity)


Similarity between the two sentences...
0.9378001689910889


### References {.unnumbered}

**SLP** Ch. 5, "Embeddings" [link](https://web.stanford.edu/~jurafsky/slp3/5.pdf)  
**SLP** Ch. 10 "Masked Language Models" [link](https://web.stanford.edu/~jurafsky/slp3/10.pdf)